# Lab 5: Decision Tree Classification

**Name:** Yashodhara Heggade  
**Reg:** 2427010154  
**Sec:** C/ Batch 1

## Objective
Create decision tree classifiers using CART, ID3, and C4.5-style gain ratio splitting. Compare accuracy, precision, recall, and F-measure (F1 score).

## Dataset: Kaggle Mushroom Classification


## 1. Import libraries


In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

## 2. Load and inspect the Kaggle dataset


In [3]:
csv_candidates = [
    Path("/kaggle/input/mushroom-classification/mushrooms.csv"),
    Path("/kaggle/input/mushroom-classification/mushroom.csv"),
    Path("mushrooms.csv"),
    Path("data/mushrooms.csv")
]
csv_path = next((path for path in csv_candidates if path.exists()), None)

if csv_path is None:
    raise FileNotFoundError(
        "Could not find mushrooms.csv. Download it from the Kaggle dataset page and "
        "place it beside this notebook (or in ./data/mushrooms.csv). On Kaggle, attach "
        "the dataset so it is available under /kaggle/input/mushroom-classification/."
    )

df = pd.read_csv(csv_path)
df = df.replace("?", "Unknown")

print("Loaded file:", csv_path)
print("Dataset shape (rows, columns):", df.shape)
print("Input attributes:", df.shape[1] - 1)
df.head()

Loaded file: mushrooms.csv
Dataset shape (rows, columns): (8124, 10)
Input attributes: 9


,class,cap-shape,cap-surface,cap-color,bruises,odor,gill-attachment,gill-spacing,gill-size,gill-color
0,p,x,s,n,t,p,f,c,n,k
1,e,x,s,y,t,a,f,c,b,k
2,e,b,s,w,t,l,f,c,b,n
3,p,x,y,w,t,p,f,c,n,n
4,e,x,s,g,f,n,f,w,b,k


In [4]:
# Check column types, missing values, and class balance
print("Missing values in each column:")
print(df.isnull().sum())
print("\nClass counts (e = edible, p = poisonous):")
print(df["class"].value_counts())

Missing values in each column:
class              0
cap-shape          0
cap-surface        0
cap-color          0
bruises            0
odor               0
gill-attachment    0
gill-spacing       0
gill-size          0
gill-color         0
dtype: int64

Class counts (e = edible, p = poisonous):
class
e    4208
p    3916
Name: count, dtype: int64


## 3. Prepare the features and split the data


In [5]:
X = df.drop(columns="class")
y = df["class"]
feature_names = list(X.columns)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

print("Training records:", len(X_train))
print("Testing records:", len(X_test))

Training records: 6093
Testing records: 2031


## 4. CART decision tree


In [6]:
cart_preprocessor = ColumnTransformer(
    transformers=[("categories", OneHotEncoder(handle_unknown="ignore"), feature_names)]
)

cart_model = Pipeline(steps=[
    ("encode", cart_preprocessor),
    ("tree", DecisionTreeClassifier(
        criterion="gini", max_depth=7, random_state=42
    ))
])
cart_model.fit(X_train, y_train)
cart_predictions = cart_model.predict(X_test)
print("CART model trained.")

CART model trained.


## 5. Implement ID3 and C4.5-style trees


In [7]:
def entropy(labels):
    """Return class entropy for a set of labels."""
    proportions = labels.value_counts(normalize=True)
    return -sum(p * np.log2(p) for p in proportions)


def information_gain(data, feature, target):
    """Return entropy reduction after splitting on a categorical feature."""
    parent_entropy = entropy(data[target])
    child_entropy = 0
    for _, group in data.groupby(feature, observed=True):
        child_entropy += (len(group) / len(data)) * entropy(group[target])
    return parent_entropy - child_entropy


def split_information(data, feature):
    """Return entropy of the groups created by a feature split."""
    proportions = data[feature].value_counts(normalize=True)
    return -sum(p * np.log2(p) for p in proportions)


def choose_feature(data, features, target, method):
    gains = {feature: information_gain(data, feature, target) for feature in features}
    if method == "ID3":
        return max(gains, key=gains.get)

    # C4.5-style: first keep features with at least average information gain
    average_gain = sum(gains.values()) / len(gains)
    candidates = [feature for feature in features if gains[feature] >= average_gain]
    if not candidates:
        candidates = features

    gain_ratios = {}
    for feature in candidates:
        split_info = split_information(data, feature)
        gain_ratios[feature] = gains[feature] / split_info if split_info > 0 else 0
    return max(gain_ratios, key=gain_ratios.get)


def build_tree(data, features, target, method, depth=0, max_depth=7, min_samples=10):
    """Recursively create an ID3 or C4.5-style categorical tree."""
    labels = data[target]
    majority_class = labels.mode()[0]

    if labels.nunique() == 1 or not features or depth >= max_depth or len(data) < min_samples:
        return {"leaf": majority_class}

    best_feature = choose_feature(data, features, target, method)
    node = {"feature": best_feature, "default": majority_class, "branches": {}}
    remaining_features = [feature for feature in features if feature != best_feature]

    for value, group in data.groupby(best_feature, observed=True):
        node["branches"][value] = build_tree(
            group, remaining_features, target, method,
            depth + 1, max_depth, min_samples
        )
    return node


def predict_one(tree, row):
    if "leaf" in tree:
        return tree["leaf"]
    value = row[tree["feature"]]
    branch = tree["branches"].get(value)
    if branch is None:  # Handle a category not seen while training
        return tree["default"]
    return predict_one(branch, row)


def predict_tree(tree, rows):
    return rows.apply(lambda row: predict_one(tree, row), axis=1)

train_data = X_train.copy()
train_data["class"] = y_train

id3_tree = build_tree(train_data, feature_names, "class", "ID3")
c45_tree = build_tree(train_data, feature_names, "class", "C4.5")
id3_predictions = predict_tree(id3_tree, X_test)
c45_predictions = predict_tree(c45_tree, X_test)
print("ID3 and C4.5-style models trained.")

ID3 and C4.5-style models trained.


## 6. Compare model metrics

Here poisonous (`p`) is the positive class for precision, recall, and F1 score. F1 is the harmonic mean of precision and recall. Accuracy is the fraction of all test records classified correctly.

In [8]:
predictions = {
    "CART": cart_predictions,
    "ID3": id3_predictions,
    "C4.5-style": c45_predictions
}

results = []
for algorithm, predicted in predictions.items():
    results.append({
        "Algorithm": algorithm,
        "Accuracy": accuracy_score(y_test, predicted),
        "Precision (poisonous)": precision_score(y_test, predicted, pos_label="p", zero_division=0),
        "Recall (poisonous)": recall_score(y_test, predicted, pos_label="p", zero_division=0),
        "F-measure / F1 (poisonous)": f1_score(y_test, predicted, pos_label="p", zero_division=0)
    })

results_df = pd.DataFrame(results).set_index("Algorithm")
results_df.round(3)

,Accuracy,Precision (poisonous),Recall (poisonous),F-measure / F1 (poisonous)
Algorithm,,,,
CART,0.995,1.000,0.990,0.995
ID3,0.997,0.995,0.999,0.997
C4.5-style,0.997,0.995,0.999,0.997


## 7. Inspect the trees

The ID3 and C4.5-style root features show the first attribute each algorithm selected. CART's depth and leaf count give a simple view of its size.

In [9]:
print("ID3 root feature:", id3_tree.get("feature", "Leaf"))
print("C4.5-style root feature:", c45_tree.get("feature", "Leaf"))
print("CART depth:", cart_model.named_steps["tree"].get_depth())
print("CART leaves:", cart_model.named_steps["tree"].get_n_leaves())

ID3 root feature: odor
C4.5-style root feature: odor
CART depth: 7
CART leaves: 15
